# TP 03 | Peut-on distinguer les lots défectueux à partir des mesures ?

**Chapitre 3. Comparer des modèles prédictifs**

Chez Nervure Composites, Camille souhaite comprendre si la température, la pression et la durée du cycle suffisent à classer les lots. Vous allez construire une carte des lots, comparer deux SVM et un réseau de neurones sur cette carte et recommander une règle d’inspection. La ligne A ou B reste disponible pour une analyse ultérieure. On l’écarte ici pour comparer les mêmes trois capteurs numériques à chaque étape.

**Avant de commencer.** Vous devez savoir lire un tableau pandas et distinguer entraînement, validation et test. Ouvrez le notebook dans VS Code avec le noyau Python de `.venv` et exécutez les cellules dans l’ordre. Les deux CSV sont fournis. Aucun fichier produit dans un TP précédent n’est nécessaire.

**À remettre.** Une note à Camille précisant la variance conservée, les limites de la carte et une règle de priorité chiffrée. Les trois défis en fin de notebook sont indépendants et facultatifs.

## 1. Mettre les trois capteurs à une échelle comparable

**Atelier 3.1 du support MLB.** Sections 1 à 2.

![Six lots projetés sur le premier axe de PCA](figures/concept-pca.svg)

Exemple avec deux capteurs : chaque lot possède deux coordonnées. Son ombre sur le premier axe principal (PC1) donne une seule valeur le long de l’axe. Dans le TP, vous appliquerez le même principe pour passer de trois mesures à deux coordonnées.

La PCA (*Principal Component Analysis*, ou analyse en composantes principales) cherche les directions où les observations **varient le plus**. Une pression qui change de 0,5 et une durée qui change de 10 sont exprimées dans des unités différentes. Une comparaison directe des écarts numériques dépendrait du choix des unités. On remplace d’abord les pressions manquantes par la médiane de l’entraînement, puis on **standardise** chaque colonne :

$$
z_{ij}=\frac{x_{ij}-\mu_j}{\sigma_j}
$$

Pour la mesure $j$ du lot $i$, on retire la moyenne puis on divise par l’écart-type. Une valeur de +1 situe cette mesure un écart-type au-dessus de la moyenne d’entraînement. Les mêmes paramètres seront utilisés pour tous les futurs lots. On conserve la température extrême du TP 01 pour que l’expérience soit reproductible. La standardisation ne corrige pas une erreur de capteur.

In [ ]:
# Apprendre le remplacement des mesures absentes et leur échelle sur les seuls lots anciens.
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

racine = Path.cwd()
if racine.name == "corrections":
    racine = racine.parent


def lire(nom):
    return pd.read_csv(
        racine / "donnees" / nom, parse_dates=["date", "label_disponible"]
    )


train = lire("lots_train.csv")
val = lire("lots_validation.csv")
assert train.label_disponible.max() < val.date.min()
mesures = ["temperature", "pression", "duree"]
preparation = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
z_train = preparation.fit_transform(train[mesures])
z_val = preparation.transform(val[mesures])
print("Dimensions :", z_train.shape, z_val.shape)
display(train[mesures].head())

### Expérience | Le choix des unités change-t-il le lot le plus proche ?

Plusieurs méthodes comparent les lots par leur distance. Dans un exemple fictif, A mesure 70 °C pour un cycle de 10 minutes, B mesure 72 °C pour 10 minutes et C mesure 70 °C pour 11 minutes. La distance euclidienne est la racine de la somme des différences au carré.

![Distance de B et C au lot A, en changeant uniquement l’unité de la durée](figures/tp03-unites.svg)

1. Calculez à la main la distance de A à B, puis de A à C en minutes. Recommencez après conversion des minutes en secondes.
2. Calculez les distances après standardisation des trois exemples, dans chacune des deux unités. Expliquez ce qui dépend de la convention de mesure.
3. Dans l’usine, sur quels lots faut-il ajuster le standardiseur ?

In [ ]:
# Changer l’unité du temps pour observer son effet sur la notion de proximité.
exemples_minutes = np.array([[70.0, 10.0], [72.0, 10.0], [70.0, 11.0]])
exemples_secondes = exemples_minutes * [1, 60]


def distances_a(tableau):
    return np.linalg.norm(tableau[1:] - tableau[0], axis=1)


print(distances_a(exemples_minutes))
# Appliquez distances_a aux secondes puis aux deux tableaux standardisés.

**Votre réponse.** Notez vos résultats et expliquez votre conclusion en une ou deux phrases.

### Manipulation | Refaire le calcul pour une pression absente

Quel score standardisé reçoit le premier lot de validation sans pression ? Retrouvez son identifiant, lisez la médiane de remplacement, puis appliquez la formule à la main en Python. Vérifiez le résultat avec la ligne correspondante de `z_val`. Pourquoi une valeur imputée ne vaut-elle pas nécessairement zéro après standardisation ?

In [ ]:
# Reconstituer la pression standardisée d’un lot à partir des paramètres d’entraînement.
position = np.flatnonzero(val["pression"].isna())[0]
j = mesures.index("pression")
mediane = preparation[0].statistics_[j]
moyenne = preparation[1].mean_[j]
ecart_type = preparation[1].scale_[j]
print(val.iloc[position]["id"], mediane, moyenne, ecart_type)
# Calculez (valeur imputée - moyenne) / écart-type, puis comparez à z_val[position, j].

**Votre réponse.** Appuyez votre conclusion sur un calcul et précisez ce qu’il signifie pour Camille.

## 2. Construire et lire la carte PCA

Imaginez que vous projetiez un nuage de points sur une règle. En tournant la règle, les ombres des points s’étalent ou se resserrent. La première composante principale (PC1) est la direction unitaire qui maximise la variance des projections. PC2 maximise ensuite la variance restante tout en étant perpendiculaire à PC1.

$$
t_{i1}=(z_i-\bar z)^\top v_1
$$

$v_1$ contient les poids des trois mesures sur PC1. $t_{i1}$ est la coordonnée du lot sur cet axe. Le **ratio de variance expliquée** rapporte la variance d’un axe à la variance totale. Deux axes donnent une carte plane. Le troisième axe contient ce que la carte perd. La PCA s’ajuste sans utiliser le verdict ; on colore les lots après projection pour examiner la séparation des classes.

### Expérience | Faire pivoter un axe avant d’utiliser PCA

Dans ce petit nuage fictif, les deux capteurs sont déjà centrés. Le cercle marque le lot, le carré son ombre sur l’axe. Le segment en pointillé est perpendiculaire à l’axe. Pour un vecteur unitaire $u$, le produit scalaire $z^T u$ donne une seule coordonnée, mesurée le long de cet axe.

![Deux projections orthogonales des mêmes quatre points](figures/tp03-ombres.svg)

Avant de calculer, choisissez la direction qui sépare le mieux les ombres. Pour le point (2, 3), calculez la coordonnée sur chaque axe. Projetez ensuite les quatre points et comparez les variances avec la même convention. Avez-vous trouvé la meilleure des deux directions proposées ou démontré qu’elle est optimale parmi toutes les directions ?

In [ ]:
# Projeter les mêmes quatre lots sur deux directions pour comparer leur dispersion.
nuage = np.array([[-2, -1], [-1, -2], [1, 0], [2, 3]])
diagonale = np.array([1, 1]) / np.sqrt(2)
perpendiculaire = np.array([-1, 1]) / np.sqrt(2)
# Calculez nuage @ direction pour chaque direction, puis .var(ddof=1).

**Votre réponse.** Notez vos résultats et expliquez votre conclusion en une ou deux phrases.

In [ ]:
# Construire les axes sur les lots anciens puis placer les nouveaux lots dans ce même repère.
pca = PCA(n_components=3, svd_solver="full")
pc_train = pca.fit_transform(z_train)
pc_val = pca.transform(z_val)
poids = pd.DataFrame(pca.components_, columns=mesures, index=["PC1", "PC2", "PC3"])
display(poids.round(3))
parts = pd.Series(pca.explained_variance_ratio_, index=poids.index)
display(pd.DataFrame({"part": parts, "cumul": parts.cumsum()}).round(3))

![Variance expliquée par axe et projection des lots d’entraînement](figures/tp03-carte-pca.svg)

Chaque point représente un lot ancien. Les couleurs et les symboles indiquent le verdict connu du laboratoire. Les axes s’expriment en coordonnées PCA, sans unité physique.

### Manipulation | Retrouver un point et chiffrer la perte

Recalculez PC1 du premier lot en multipliant chaque mesure centrée par le poids correspondant, puis en additionnant les trois contributions. Combien de variance resterait invisible sur la carte ? D’après le graphique, peut-on fixer une frontière parfaite à l’œil ?

In [ ]:
# Décomposer la coordonnée du premier lot en contributions de ses trois capteurs.
centre = z_train[0] - pca.mean_
direction = pca.components_[0]
# Multipliez les trois valeurs par leurs poids. Comparez leur somme à pc_train[0, 0].
# Calculez 1 - la somme des deux premières parts.

**Votre réponse.** Appuyez votre conclusion sur un calcul et précisez ce qu’il signifie pour Camille.

### Enquête | Retrouver les mesures à partir de la carte

![Projection du point (2, 0) sur la droite diagonale, avec le résidu perpendiculaire.](figures/tp03-retour-plan.svg)

Dans cet exemple à deux dimensions, le point projeté est (1, 1). Quel autre point entier, de l’autre côté de la droite, aurait la même projection ? Indice : réfléchissez le point (2, 0) par rapport à la diagonale. Reprenez ensuite le raisonnement sur les trois capteurs du lot réel.

Camille désigne un point sur la carte et demande sa température. Pour revenir vers les mesures, on combine les directions avec les coordonnées du lot puis on annule la standardisation. Deux coordonnées sur trois ne déterminent plus un lot de façon unique. En posant PC3 à zéro, on obtient la projection sur le plan PCA.

Sur le premier lot de validation, comparez la température et la durée reconstruites à partir de deux axes aux mesures enregistrées. Essayez ensuite les trois axes. Pourquoi une pression initialement absente ne peut-elle pas être récupérée, même avec trois axes ?

In [ ]:
# Revenir aux unités des capteurs pour mesurer ce que la carte à deux axes laisse de côté.
coordonnees = pc_val[[0]].copy()


def retour_capteurs(coord):
    standardise = pca.inverse_transform(coord)
    return preparation[1].inverse_transform(standardise)[0]


# Travaillez sur une copie des coordonnées. Annulez uniquement la troisième.
# Comparez avec val.iloc[0][mesures], en conservant les unités de chaque capteur.

**Votre réponse.** Notez vos résultats et expliquez votre conclusion en une ou deux phrases.

## 3. Apprendre une frontière avec un SVM

**Atelier 3.2 du support MLB.** Sections 3 à 5.

Une **machine à vecteurs de support** (*support vector machine*, SVM) apprend une frontière à partir des verdicts.

![Frontière et vecteurs de support d’un SVM sur six lots](figures/concept-svm.svg)

La droite centrale sépare les deux classes. Les deux lots cerclés touchent les bords de la marge et déterminent ici la position de la droite. Cet exemple séparable prépare le cas des lots qui se recouvrent.

Pour un noyau linéaire, $t_1$ et $t_2$ sont les coordonnées PCA du lot, $w_1$ et $w_2$ les poids appris, et $b$ la constante. Le score est

$$
f(t)=w_1t_1+w_2t_2+b.
$$

La droite $f(t)=0$ sépare les deux décisions par défaut. Les droites $f(t)=-1$ et $f(t)=1$ délimitent une bande de largeur $2/\lVert w\rVert$. L’entraînement cherche une marge large en pénalisant les violations de cette bande. Les **vecteurs de support** sont les observations qui contribuent à définir la frontière. Augmenter $C$ accroît le poids des violations dans le compromis.

Le noyau **RBF** (*radial basis function*) compare deux lots par leur proximité :

$$
K(t,u)=\exp(-\gamma\lVert t-u\rVert^2).
$$

Un grand $\gamma$ réduit la portée de cette proximité. Le SVM peut alors tracer une frontière courbe. Nous fixons $C=1$ et laissons scikit-learn calculer $\gamma$ sur l’entraînement avec `gamma="scale"`. Le score signé donné par `decision_function` classe les lots. Il se lit par rapport à un seuil et ne constitue pas une probabilité.

### Calcul à la main | Être bien classé et respecter la marge

Pour isoler la géométrie, prenons un modèle fictif dont le score est la coordonnée horizontale $f(x)=x$. Pour ce calcul, codez un défaut par $y=+1$ et un lot conforme par $y=-1$. Le produit $y f(x)$ est positif quand le signe prédit est correct. La **perte charnière** (*hinge loss*) vaut $\max(0,1-y f(x))$. Elle pénalise aussi un point correctement classé qui reste dans la bande.

![Frontière verticale, bande de marge et trois lots A, B, C](figures/tp03-marge.svg)

A et B sont défectueux (cercles), C est conforme (croix). Leurs scores sont 1,5, 0,4 et 0,6. Pour chacun, prévoyez la décision, la perte puis le statut : hors de la bande, dans la bande du bon côté ou mal classé. Quel lot serait invisible si on comptait seulement les erreurs ?

In [ ]:
# Distinguer une erreur de classe d’une décision correcte située à l’intérieur de la marge.
scores_exemple = np.array([1.5, 0.4, 0.6])
classes_exemple = np.array([1, 1, -1])
# Calculez le produit terme à terme. np.maximum(0, ...) applique le plancher à zéro.

**Votre réponse.** Notez vos résultats et expliquez votre conclusion en une ou deux phrases.

In [ ]:
# Entraîner les deux SVM sur la même carte pour isoler l’effet du choix de noyau.
modeles = {
    "Linéaire": SVC(kernel="linear", C=1),
    "RBF": SVC(kernel="rbf", C=1, gamma="scale"),
}
for modele in modeles.values():
    modele.fit(pc_train[:, :2], train["defaut"])
scores = {
    nom: modele.decision_function(pc_val[:, :2]) for nom, modele in modeles.items()
}
display(pd.DataFrame(scores, index=val["id"]).head().round(3))

![Frontières de décision linéaire et RBF sur la même carte des lots](figures/tp03-frontieres.svg)

Les traits noirs marquent le seuil zéro des deux modèles. Les pointillés marquent les niveaux de score −1 et +1 lorsqu’ils traversent la fenêtre affichée. Les deux panneaux utilisent la même échelle. Le cadrage privilégie la zone où se trouvent la plupart des lots.

### Expérience | Jusqu’où un lot voisin influence-t-il le noyau ?

À distance nulle, deux lots ont une similarité RBF de 1. Quand la distance augmente, la similarité décroît. Le paramètre gamma fixe la rapidité de la décroissance.

![Similarité RBF en fonction de la distance pour trois valeurs de gamma](figures/tp03-portee.svg)

Lisez la similarité à distance 1 pour gamma = 0,25 et gamma = 4. Vérifiez en Python. Calculez ensuite la distance à laquelle la similarité tombe à 0,5 pour chaque réglage, en résolvant $\exp(-\gamma d^2)=0{,}5$. Pourquoi un gamma élevé peut-il rendre la frontière plus locale ?

In [ ]:
# Relier la valeur de gamma à la portée de la similarité entre deux lots.
gammas = np.array([0.25, 1.0, 4.0])
distance = 1.0
# np.exp donne l’exponentielle. Utilisez np.log(2) pour trouver la demi-portée.

**Votre réponse.** Notez vos résultats et expliquez votre conclusion en une ou deux phrases.

### Manipulation | Enquêter sur un désaccord

À seuil zéro, repérez un lot recevant deux décisions différentes. Affichez ses mesures et ses deux scores. Identifiez le modèle qui aurait déclenché une inspection et consultez ensuite le verdict pour évaluer les deux décisions. Pourquoi ce seul lot ne suffit-il pas à choisir le modèle ?

In [ ]:
# Repérer les lots pour lesquels les deux frontières conduisent à des décisions opposées.
alertes = pd.DataFrame(scores).ge(0)
desaccord = alertes["Linéaire"] != alertes["RBF"]
# np.flatnonzero(desaccord) donne les positions à comparer avec val.iloc.

**Votre réponse.** Appuyez votre conclusion sur un calcul et précisez ce qu’il signifie pour Camille.

## 4. Faire un choix compatible avec la capacité de l’atelier

Sur 240 lots, Camille dispose de 48 inspections. Comme au TP 02, une fausse alerte coûte un point, un défaut manqué en coûte dix. Pour chaque modèle, on essaie des seuils tirés de ses **scores d’entraînement** aux quantiles 80 %, 85 %, 90 % et 95 %. Un seuil au 80e percentile vise environ les 20 % de scores les plus élevés sur les lots anciens. La proportion sur une période ultérieure peut changer.

La validation sert à choisir parmi ces huit règles et la référence « aucune inspection ». On écarte d’abord les règles qui dépassent la capacité, puis on minimise le coût. À égalité, on préfère la charge la plus faible. Le test final reste réservé au TP 08.

In [ ]:
# Évaluer les seuils issus des lots anciens avec la capacité et les coûts de l’atelier.
def evaluer(nom, score, seuil):
    decision = score >= seuil
    _vn, fp, fn, vp = confusion_matrix(val["defaut"], decision, labels=[0, 1]).ravel()
    return {
        "modèle": nom,
        "seuil": seuil,
        "inspections": fp + vp,
        "VP": vp,
        "FP": fp,
        "FN": fn,
        "coût": fp + 10 * fn,
    }


# Construire les seuils avec les scores d’entraînement pour simuler leur usage ultérieur.
def candidats(nom, modele, x_train, x_val):
    anciens = modele.decision_function(x_train)
    nouveaux = modele.decision_function(x_val)
    return [
        evaluer(nom, nouveaux, np.quantile(anciens, q))
        for q in [0.80, 0.85, 0.90, 0.95]
    ]


def classer(lignes):
    tableau = pd.DataFrame(lignes)
    return tableau.loc[tableau["inspections"] <= 48].sort_values(
        ["coût", "inspections"], kind="stable"
    )


# Inclure la décision de ne lancer aucune inspection comme point de comparaison.
reference = evaluer("Aucune inspection", np.zeros(len(val)), 1)
essais = [reference]
for nom, modele in modeles.items():
    essais.extend(candidats(nom, modele, pc_train[:, :2], pc_val[:, :2]))
display(pd.DataFrame(essais).round(3))

### Décision | Que vaut une place d’inspection supplémentaire ?

Chaque symbole ci-dessous représente une règle du tableau précédent. En abscisse, on lit la charge ; en ordonnée, le coût à minimiser. Avec 48 places, seules les règles sur la ligne verticale ou à sa gauche sont réalisables.

![Nuage des règles candidates, coût et charge avec contrainte de 48 inspections](figures/tp03-budget.svg)

La maintenance hésite entre 40, 48 et 56 places pour la même période. Sans réentraîner les modèles, cherchez pour chaque budget la meilleure règle disponible. À quel moment la recommandation change-t-elle ? Expliquez pourquoi le gain peut être nul même quand le nombre de places augmente.

In [ ]:
# Comparer les règles réalisables quand le nombre de créneaux d’inspection change.
catalogue = pd.DataFrame(essais)
budgets = [40, 48, 56]
# Pour chaque budget, filtrez inspections <= budget, triez par coût puis inspections.
# Gardez la première ligne. La référence sans inspection reste disponible.

**Votre réponse.** Notez vos résultats et expliquez votre conclusion en une ou deux phrases.

### Décision intermédiaire | Quel SVM recommander à Camille ?

Avant de lancer le classement, rayez les règles irréalisables dans le tableau et repérez votre favorite. Donnez le noyau, le seuil inclusif, le nombre de contrôles, de défauts interceptés et manqués, le coût et le gain par rapport à la référence. Distinguez la frontière du dessin et le seuil choisi pour l’atelier.

Gardez cette première décision comme point de comparaison. Dans la section suivante, le MLP reçoit les mêmes coordonnées PCA. Pour savoir s’il apporte quelque chose, reprenez le coût du meilleur SVM et comparez-le au coût du MLP sur les mêmes lots de validation, avec la même limite de places.

In [ ]:
# Retenir une règle admissible et chiffrer son gain par rapport à l’absence d’inspection.
admissibles = classer(essais)
# Lisez admissibles.iloc[0] puis comparez son coût à celui de reference.
# Gardez la précision complète du seuil dans le code, arrondissez uniquement pour le compte rendu.

**Votre réponse.** Appuyez votre conclusion sur un calcul et précisez ce qu’il signifie pour Camille.

## 5. Un petit réseau apprend-il une meilleure règle ?

Un **perceptron multicouche** (MLP) combine ici les deux coordonnées PCA dans huit neurones cachés. Chaque neurone pondère les entrées, ajoute un biais et applique une fonction dite ReLU qui ramène les valeurs négatives à zéro. La couche de sortie combine les huit réponses et les transforme en score entre 0 et 1.

![Deux axes PCA alimentent la couche cachée puis le score de défaut](figures/tp03-reseau.svg)

Suivez un point de gauche à droite. Pourquoi huit neurones peuvent-ils répondre à des zones différentes de la carte ? Pendant l’entraînement, la rétropropagation calcule comment chaque poids contribue à l’erreur. L’optimiseur ajuste ces poids sur les verdicts des lots anciens. La validation sert ensuite à comparer les règles.

Entraînez le réseau sur la même projection que les deux SVM. Pour comparer leurs décisions, reprenez les mêmes quantiles des scores du train, le budget de 48 places et le coût du TP 02. Relevez le meilleur seuil admissible par modèle et expliquez votre choix à Camille.

In [ ]:
# Entraîner huit neurones sur les seuls lots anciens, puis scorer les deux périodes.
from sklearn.neural_network import MLPClassifier

reseau = MLPClassifier(
    hidden_layer_sizes=(8,), solver="lbfgs", max_iter=1500, random_state=42
)
reseau.fit(pc_train[:, :2], train.defaut)
score_train = reseau.predict_proba(pc_train[:, :2])[:, 1]
score_val = reseau.predict_proba(pc_val[:, :2])[:, 1]
regles_mlp = [
    evaluer("MLP", score_val, np.quantile(score_train, q))
    for q in (0.80, 0.85, 0.90, 0.95)
]
comparatif = classer(essais + regles_mlp).drop_duplicates("modèle")
display(comparatif.round(3))

![Frontière du MLP et lots anciens dans le plan PCA](figures/tp03-frontiere-mlp.svg)

Le trait représente le seuil de 0,50. Où se courbe-t-il ? Que devient la zone inspectée quand Camille augmente le seuil ? Comparez ensuite les défauts manqués et les fausses alertes dans le tableau.

**Votre recommandation.** Quel modèle retenez-vous sur cette grille ? Chiffrez le compromis entre places utilisées et défauts manqués.

### Manipulation | Retrouver le score d’un lot

Choisissez L0721 sur la carte. Calculez les huit activations à partir des poids appris. Combien sont positives ? Combinez-les avec les poids de sortie puis appliquez la fonction logistique. Comparez à la prédiction du réseau. Un neurone à zéro contribue-t-il au score brut de ce lot ?

![Activation pour deux valeurs du biais](figures/tp03-relu-biais.svg)

Sur cette coupe fictive du plan PCA, le poids de PC1 vaut 1. À PC1 = 0,5, quel neurone répond encore ? Avec un biais nul, la ReLU transmet 0,5. Avec un biais de −1, elle reçoit −0,5 et transmet zéro. Dans le réseau entraîné, chaque neurone apprend sa propre orientation et son propre seuil d’activation.

In [ ]:
# Reconstituer la propagation vers la sortie pour un seul lot.
position_mlp = np.flatnonzero(val.id.eq("L0721"))[0]
point = pc_val[position_mlp, :2]
poids_entree, poids_sortie = reseau.coefs_
biais_cache, biais_sortie = reseau.intercepts_
activations = np.maximum(0, point @ poids_entree + biais_cache)
print("Activations :", activations.round(3))
# Combinez les activations et comparez au score prédit.

**Votre réponse.** Notez le nombre de neurones actifs et expliquez le résultat du contrôle.

### Enquête | Quelle erreur le réseau cherche-t-il à réduire ?

![Perte pour chaque verdict selon le score annoncé](figures/tp03-perte.svg)

Pour un lot défectueux, lisez la courbe à 0,20 puis à 0,80. Une erreur commise avec assurance coûte davantage au réseau. Si $y$ désigne le verdict (0 ou 1) et $p$ le score de défaut, la perte logarithmique par lot vaut

$$
L=-y\log(p)-(1-y)\log(1-p).
$$

Pour comprendre une étape de rétropropagation, imaginez que l’on ajuste seulement le biais de sortie. Sa dérivée pour un lot vaut $p-y$. Calculez-la pour les deux scores et expliquez dans quel sens un pas de descente modifierait le biais. Pourquoi Camille compte-t-elle encore les inspections après cet entraînement ?

In [ ]:
# Isoler deux prédictions fictives pour le même verdict connu.
scores_exemple = np.array([0.20, 0.80])
verdict = 1
# Utilisez np.log et calculez la dérivée pour chaque score.
# Complétez le calcul demandé, puis interprétez le résultat.

**Votre réponse.** Notez les valeurs obtenues et justifiez votre interprétation pour Camille.

### Transférer le SVM à la lecture de chiffres

![Deux chiffres manuscrits de la collection digits](figures/tp03-chiffres.svg)

Sur un bordereau, confondre un 3 et un 8 peut envoyer un lot vers le mauvais quai. Comptez les lignes et colonnes de pixels sur les exemples agrandis. Ici, une image fournit 64 intensités de même échelle et l’étiquette identifie le chiffre écrit. Les données manuscrites de la collection UCI sont fournies avec scikit-learn. Deux SVM reçoivent le même découpage stratifié, qui conserve la proportion des chiffres. Quel sens d’erreur domine pour chaque noyau ? Pour évaluer le transfert à une nouvelle usine, que changeriez-vous dans ce découpage ?

In [ ]:
# Comparer deux noyaux sur des images réservées, avec la même échelle de pixels.
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

chiffres = load_digits()
selection = np.isin(chiffres.target, [3, 8])
pixels_train, pixels_val, chiffres_train, chiffres_val = train_test_split(
    chiffres.data[selection] / 16,
    chiffres.target[selection],
    test_size=0.3,
    random_state=42,
    stratify=chiffres.target[selection],
)
for noyau in ("linear", "rbf"):
    lecteur = SVC(kernel=noyau).fit(pixels_train, chiffres_train)
    tableau = confusion_matrix(chiffres_val, lecteur.predict(pixels_val), labels=[3, 8])
    print(noyau, "| lignes : chiffres écrits 3, 8 ; colonnes : chiffres lus 3, 8")
    print(tableau)

**Votre réponse.** Notez votre décision et la mesure qui la justifie.

## Défi facultatif | Le troisième axe contient-il un signal utile ?

Refaites l’essai RBF avec les trois composantes. Gardez les mêmes lots, la même grille de quantiles et la même pénalité. Comparez les meilleurs coûts admissibles des deux variantes. Expliquez pourquoi le pourcentage de variance expliquée ne pouvait pas prédire la réponse.

In [ ]:
# Réintroduire le troisième axe pour vérifier son utilité dans la détection des défauts.
rbf_complet = SVC(kernel="rbf", C=1, gamma="scale")
# Ajustez sur pc_train entier. Réutilisez candidats et classer.
# Pour le témoin à deux axes, filtrez admissibles sur le modèle RBF.

**Votre réponse.** Appuyez votre conclusion sur un calcul et précisez ce qu’il signifie pour Camille.

## Défi facultatif | Quel effet a la pénalisation des violations de marge ?

Sur la carte à deux axes, entraînez trois SVM linéaires avec $C$ égal à 0,01, 1 puis 100. Pour chacun, calculez la largeur de la bande de marge, le nombre de vecteurs de support et l’AUC ROC en validation. L’AUC mesure la capacité à ordonner un lot défectueux devant un lot conforme. Une valeur de 0,5 correspond à un classement aléatoire en moyenne. Une meilleure séparation géométrique garantit-elle une meilleure AUC ?

In [ ]:
# Faire varier la pénalisation pour observer ensemble la marge et la qualité du classement.
penalites = [0.01, 1, 100]
# SVC(kernel="linear", C=c) fournit coef_[0] après fit.
# np.linalg.norm calcule la norme du vecteur. support_ contient les indices de support.
print("AUC du modèle de référence :", roc_auc_score(val["defaut"], scores["Linéaire"]))

**Votre réponse.** Appuyez votre conclusion sur un calcul et précisez ce qu’il signifie pour Camille.

## Défi facultatif | Deux prototypes correspondent-ils aux deux verdicts ?

Camille se demande si les îlots visibles sur la carte représentent des types de lots. **K-means** place des centres appelés prototypes en minimisant la somme des distances au carré de chaque point à son centre le plus proche (l’inertie). Il utilise uniquement les coordonnées.

![Lots de validation répartis par proximité des centres appris sur les lots anciens](figures/tp03-prototypes.svg)

La couleur indique le centre le plus proche. Les croix sont les centres ajustés sur les lots anciens. Les verdicts sont masqués pour cette première lecture.

Ajustez deux puis trois prototypes sur la carte d’entraînement. Affectez chaque lot de validation au centre le plus proche. Pour chaque groupe, comptez les lots et calculez la fréquence des défauts. Peut-on interpréter un numéro de groupe comme un verdict ? La baisse de l’inertie suffit-elle pour retenir trois prototypes ?

In [ ]:
# Comparer deux découpages du nuage avant de consulter les verdicts au sein des groupes.
tailles = [2, 3]
# KMeans(n_clusters=k, n_init=10, random_state=42) rend l’essai reproductible.
# Ajustez sur pc_train[:, :2], puis utilisez predict sur pc_val[:, :2].
# Regroupez le verdict de validation par les numéros obtenus.
prototype_exemple = KMeans(n_clusters=2, n_init=10, random_state=42)

**Votre réponse.** Appuyez votre conclusion sur un calcul et précisez ce qu’il signifie pour Camille.

### Pour approfondir

Les exemples de ce TP utilisent les définitions et les implémentations décrites dans la documentation scikit-learn. [PCA](https://scikit-learn.org/stable/modules/decomposition.html#pca) | [SVM](https://scikit-learn.org/stable/modules/svm.html) | [K-means](https://scikit-learn.org/stable/modules/clustering.html#k-means)

[Ouvrir le corrigé expliqué](corrections/TP03_pca_et_svm.ipynb)